# Notebook 04 — Exponential and Logistic Growth

**Module:** 02 — Mathematics for Biology  
**Notebook:** 04 of 12  
**Prerequisites:** NB02, NB03  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Two models describe nearly every growth process in biology. Exponential growth:
what happens when resources are unlimited — bacteria in a new flask, a virus early
in an epidemic, a tumour cell lineage. Logistic growth: what happens when resources
are finite — that same population hitting a carrying capacity, a chemostat culture
at steady state. Both are ODEs you can solve analytically — and both are fitted
to real data every day in biology labs.

---
## Step 2 — Intuition

**Exponential:** each cell divides at rate $r$. The more cells, the faster the total
population grows. No ceiling. On a log scale, straight line.

**Logistic:** same idea, but the growth rate shrinks as the population approaches
the carrying capacity $K$. When $N = K$, births exactly balance deaths. S-shaped
curve on a linear scale.

The logistic model is exponential growth with a brake.

---
## Step 3 — Biological Background

**Bacterial growth phases:**
- **Lag phase:** cells adapt to medium; effectively $dN/dt \approx 0$
- **Exponential (log) phase:** $dN/dt = rN$; this is the growth rate biologists report
- **Stationary phase:** resources limited; $N \approx K$
- **Death phase:** $N$ declines below $K$

The logistic model captures exponential + stationary phases but not lag or death.

**Tumor growth:** solid tumours often follow logistic growth — exponential early
(avascular phase), then plateau when diffusion limits oxygen/nutrient supply.

**Doubling time:** $t_{1/2} = \ln(2)/r$ — the time for population to double.

---
## Step 4 — Mathematical Explanation

**Exponential growth ODE:**
$$\frac{dN}{dt} = rN, \quad N(0) = N_0$$

**Analytical solution** (separate variables, integrate both sides):
$$\frac{dN}{N} = r\,dt \implies \ln N = rt + C \implies N(t) = N_0 e^{rt}$$

**Logistic growth ODE:**
$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right), \quad N(0) = N_0$$

**Analytical solution** (partial fractions):
$$N(t) = \frac{K}{1 + \left(\frac{K}{N_0} - 1\right)e^{-rt}}$$

**Fixed points** (where $dN/dt = 0$):
- $N^* = 0$ (unstable — any positive perturbation grows away)
- $N^* = K$ (stable — perturbations return to $K$)

---
## Step 5 — Computational Explanation

To fit either model to experimental data:
1. Define the model function $N(t; r, K, N_0)$
2. Provide observed $(t_i, N_i)$ pairs
3. Use `scipy.optimize.curve_fit` to find the $(r, K, N_0)$ that minimise sum-of-squared residuals
4. Check the fit visually and report 95% confidence intervals from the covariance matrix

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Exponential growth: model and analytical solution
def exponential_growth(t: np.ndarray, r: float, N0: float) -> np.ndarray:
    """Analytical solution to dN/dt = rN."""
    return N0 * np.exp(r * t)

# E. coli: r = ln(2)/20 min
r_ecoli = np.log(2) / 20  # per minute
t = np.linspace(0, 120, 500)  # 2 hours
N_exp = exponential_growth(t, r_ecoli, N0=1.0)

print(f"E. coli growth rate r = {r_ecoli:.4f} per minute")
print(f"Doubling time: {np.log(2)/r_ecoli:.1f} minutes")
print(f"N at t=60 min: {exponential_growth(60, r_ecoli, 1):.2f} cells")
print(f"N at t=120 min: {exponential_growth(120, r_ecoli, 1):.2f} cells")

In [ ]:
# Cell 6.2 — Logistic growth: model and analytical solution
def logistic_growth(t: np.ndarray, r: float, K: float, N0: float) -> np.ndarray:
    """Analytical solution to dN/dt = rN(1 - N/K)."""
    return K / (1 + (K/N0 - 1) * np.exp(-r * t))

K = 1e6   # carrying capacity
N0 = 1.0  # initial
N_log = logistic_growth(t, r_ecoli, K, N0)

# Verify fixed points
dN_dt_logistic = lambda N: r_ecoli * N * (1 - N/K)
print("Fixed point verification:")
print(f"  dN/dt at N=0: {dN_dt_logistic(0):.4f}")
print(f"  dN/dt at N=K: {dN_dt_logistic(K):.6f}")
print(f"  dN/dt at N=K/2: {dN_dt_logistic(K/2):.4f}  (maximum growth rate)")

In [ ]:
# Cell 6.3 — Curve fitting: recover parameters from noisy data
rng = np.random.default_rng(42)

# Generate synthetic 'experimental' data
r_true, K_true, N0_true = 0.03, 8e5, 2.0
t_obs = np.array([0, 10, 20, 30, 40, 50, 60, 80, 100, 120], dtype=float)
N_true = logistic_growth(t_obs, r_true, K_true, N0_true)
N_obs = N_true * rng.lognormal(0, 0.15, size=len(t_obs))  # 15% log-normal noise

# Fit logistic model
p0 = [0.02, 5e5, 5.0]  # initial guesses
bounds = ([0, 0, 0], [1, 1e8, 100])
popt, pcov = curve_fit(logistic_growth, t_obs, N_obs, p0=p0, bounds=bounds, maxfev=10000)
r_fit, K_fit, N0_fit = popt
r_err, K_err, N0_err = np.sqrt(np.diag(pcov))

print(f"{'Parameter':<12} {'True':>10} {'Fitted':>12} {'95% CI (±2σ)':>20}")
print("-" * 58)
print(f"  r          {r_true:>10.4f} {r_fit:>12.4f} {2*r_err:>14.4f}")
print(f"  K          {K_true:>10.0f} {K_fit:>12.0f} {2*K_err:>14.0f}")
print(f"  N0         {N0_true:>10.1f} {N0_fit:>12.2f} {2*N0_err:>14.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Three-panel growth model figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: linear scale comparison
ax = axes[0]
ax.plot(t, N_exp / 1e6, color="tomato", lw=2, label="Exponential")
ax.plot(t, N_log / 1e6, color="steelblue", lw=2, label="Logistic")
ax.axhline(K/1e6, color="steelblue", ls="--", alpha=0.5, label=f"K = {K:.0e}")
ax.set_xlabel("Time (min)"); ax.set_ylabel("N (millions)")
ax.set_title("Exponential vs. logistic")
ax.legend(); ax.set_ylim(bottom=0)

# Panel 2: log scale — exponential is linear
ax = axes[1]
ax.semilogy(t, N_exp, color="tomato", lw=2, label="Exponential")
ax.semilogy(t, N_log, color="steelblue", lw=2, label="Logistic")
ax.set_xlabel("Time (min)"); ax.set_ylabel("N (log scale)")
ax.set_title("Log scale: exponential = straight line")
ax.legend()

# Panel 3: curve fit result
ax = axes[2]
t_dense = np.linspace(0, 130, 300)
ax.scatter(t_obs, N_obs, color="black", s=25, zorder=5, label="Observed")
ax.plot(t_dense, logistic_growth(t_dense, r_true, K_true, N0_true),
        color="gray", ls="--", lw=1.5, label="True model")
ax.plot(t_dense, logistic_growth(t_dense, *popt),
        color="steelblue", lw=2, label="Fitted")
ax.set_xlabel("Time (min)"); ax.set_ylabel("N")
ax.set_title("Logistic growth curve fit")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Derive the logistic growth solution $N(t) = K / (1 + (K/N_0 - 1)e^{-rt})$
   analytically using separation of variables and partial fractions.
2. A bacterial culture has $r = 0.05$/min and $K = 10^9$ cells. Starting at $N_0 = 100$,
   what time does it reach 90% of $K$? Solve analytically (rearrange the logistic formula
   for $t$), then verify numerically.
3. Fit an exponential model (not logistic) to the same synthetic data from Cell 6.3
   using only the first 5 time points (where the population is far below K). Do the
   recovered parameters match the true logistic parameters? Why or why not?
4. What is $dN/dt$ when $N = K/2$? Show that this is the maximum growth rate for
   the logistic model.

---
## Quiz — Active Recall

1. Write the logistic ODE from memory. Define every symbol.
2. What are the two fixed points of the logistic model? Which is stable and which is unstable?
3. Why does the logistic model produce an S-shaped curve?
4. What is a doubling time? Write its formula in terms of $r$.
5. If you plot $\ln N$ vs. $t$ for exponential growth, what shape do you get? Why?

---
## Reflection

**Date completed:** ____________________

> *[Could you derive the logistic solution from scratch? Can you distinguish exponential from logistic on a log-scale plot?]*

---
*Next: `05_difference_vs_differential_equations.ipynb`*